<a href="https://colab.research.google.com/github/br05758135-cell/gerador-documentos-disciplinares/blob/main/teste_game.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import random

numero_secreto = random.randint(1, 10)

while True:
    tentativa = int(input("Adivinhe um número de 1 a 10: "))

    if tentativa == numero_secreto:
        print("🎉 Você acertou!")
        break
    elif tentativa < numero_secreto:
        print("⬆️ Mais alto!")
    else:
        print("⬇️ Mais baixo!")

KeyboardInterrupt: Interrupted by user

In [ ]:
import random

opcoes = ["pedra", "papel", "tesoura"]

jogador = input("Escolha pedra, papel ou tesoura: ").lower()
computador = random.choice(opcoes)

print("Computador escolheu:", computador)

if jogador == computador:
    print("Empate!")
elif (
    (jogador == "pedra" and computador == "tesoura") or
    (jogador == "papel" and computador == "pedra") or
    (jogador == "tesoura" and computador == "papel")
):
    print("Você venceu!")
else:
    print("Você perdeu!")

In [10]:
from IPython.display import HTML

HTML("""
<style>
  body { background:#050509; }
  #gameBox {
    width:980px;
    margin:10px auto;
    border:3px solid #30224e;
    box-shadow:0 0 45px #1b0f42;
    background:#050509;
  }
  canvas { display:block; background:#07070d; }
  #info {
    width:980px;
    margin:8px auto;
    color:#ddd;
    font-family:Arial, sans-serif;
    font-size:14px;
    line-height:1.6;
  }
  kbd {
    background:#222;
    border:1px solid #555;
    border-radius:4px;
    padding:2px 6px;
  }
</style>

<div id="gameBox">
  <canvas id="game" width="980" height="560"></canvas>
</div>

<div id="info">
  <b>Controles:</b>
  <kbd>A</kbd>/<kbd>D</kbd> mover |
  <kbd>W</kbd>/<kbd>Espaço</kbd> pular |
  <kbd>J</kbd> espada |
  <kbd>K</kbd> dash |
  <kbd>U</kbd> Amaterasu |
  <kbd>I</kbd> Kirin |
  <kbd>O</kbd> Susanoo |
  <kbd>H</kbd>→<kbd>J</kbd>→<kbd>K</kbd>→<kbd>L</kbd> Arco de Indra |
  <kbd>R</kbd> reiniciar
</div>

<script>
const canvas = document.getElementById("game");
const ctx = canvas.getContext("2d");

const W = canvas.width;
const H = canvas.height;

let keys = {};
let pressed = {};
let combo = [];

let state = "menu";
let difficulty = "normal";
let difficultyData = {
  facil:  { enemyHp:0.75, enemySpd:0.85, dmg:0.75, bossHp:0.75, name:"FÁCIL" },
  normal: { enemyHp:1.00, enemySpd:1.00, dmg:1.00, bossHp:1.00, name:"NORMAL" },
  dificil:{ enemyHp:1.35, enemySpd:1.25, dmg:1.35, bossHp:1.45, name:"DIFÍCIL" }
};

let audioCtx = null;
let musicOn = false;
let musicTimer = null;

function initAudio(){
  if(audioCtx) return;
  audioCtx = new (window.AudioContext || window.webkitAudioContext)();
}

function beep(freq=440, dur=0.08, type="sine", gain=0.04){
  if(!audioCtx) return;
  let osc = audioCtx.createOscillator();
  let g = audioCtx.createGain();
  osc.type = type;
  osc.frequency.value = freq;
  g.gain.value = gain;
  osc.connect(g);
  g.connect(audioCtx.destination);
  osc.start();
  osc.stop(audioCtx.currentTime + dur);
}

function startMusic(){
  if(!audioCtx || musicOn) return;
  musicOn = true;
  let notes = [110, 146.83, 164.81, 196, 174.61, 130.81];
  let i = 0;
  musicTimer = setInterval(() => {
    if(state === "playing" || state === "bossIntro" || state === "victoryCutscene"){
      beep(notes[i % notes.length], 0.18, "triangle", 0.018);
      if(i % 4 === 0) beep(notes[(i+2) % notes.length]/2, 0.28, "sine", 0.012);
      i++;
    }
  }, 520);
}

document.addEventListener("keydown", e => {
  const k = e.key.toLowerCase();

  if(!keys[k]) pressed[k] = true;
  keys[k] = true;

  if(e.key === " "){
    if(!keys[" "]) pressed[" "] = true;
    keys[" "] = true;
    e.preventDefault();
  }

  if(state === "menu"){
    if(k === "1") startGame("facil");
    if(k === "2") startGame("normal");
    if(k === "3") startGame("dificil");
  }

  if(k === "r"){
    resetGame();
    state = "menu";
  }

  combo.push(k);
  if(combo.length > 8) combo.shift();
});

document.addEventListener("keyup", e => {
  const k = e.key.toLowerCase();
  keys[k] = false;
  if(e.key === " ") keys[" "] = false;
});

canvas.addEventListener("click", e => {
  if(state !== "menu") return;

  const rect = canvas.getBoundingClientRect();
  const mx = e.clientX - rect.left;
  const my = e.clientY - rect.top;

  if(mx > 340 && mx < 640 && my > 245 && my < 295) startGame("facil");
  if(mx > 340 && mx < 640 && my > 310 && my < 360) startGame("normal");
  if(mx > 340 && mx < 640 && my > 375 && my < 425) startGame("dificil");
});

let world = { width:3600, height:760 };
let cam = { x:0 };

let player, platforms, spikes, enemies, boss;
let particles = [];
let slashes = [];
let flames = [];
let lightning = [];
let arrows = [];
let cubes = [];
let rods = [];
let orbs = [];
let texts = [];
let leaves = [];

let bossIntro = { timer:0, done:false };
let victoryTimer = 0;
let stormFlash = 0;

function startGame(diff){
  difficulty = diff;
  initAudio();
  startMusic();
  resetGame();
  state = "playing";
  beep(220,0.12,"triangle",0.06);
  beep(440,0.12,"triangle",0.04);
}

function resetGame(){
  const d = difficultyData[difficulty];

  state = state === "menu" ? "menu" : "playing";

  bossIntro = { timer:0, done:false };
  victoryTimer = 0;
  stormFlash = 0;
  combo = [];

  player = {
    x:80, y:292, w:40, h:68,
    vx:0, vy:0,
    speed:0.82, maxSpeed:5.6,
    jump:-13.6,
    facing:1,
    grounded:false,
    doubleJump:true,

    hp:10, maxHp:10,
    chakra:62, maxChakra:100,
    susanoo:0, susanooMax:100,
    susanooActive:false,
    susanooTimer:0,
    susanooDuration:560,

    inv:0,
    atkCd:0,
    dashCd:0,
    dash:0,
    amaCd:0,
    kirinCd:0,
    indraCd:0
  };

  platforms = [
    {x:0,y:470,w:670,h:90},
    {x:750,y:425,w:285,h:44},
    {x:1120,y:370,w:320,h:44},
    {x:1530,y:470,w:450,h:90},
    {x:2080,y:405,w:300,h:44},
    {x:2450,y:470,w:430,h:90},
    {x:2970,y:430,w:590,h:130},

    {x:360,y:345,w:175,h:28},
    {x:800,y:290,w:155,h:28},
    {x:1250,y:255,w:180,h:28},
    {x:1680,y:340,w:155,h:28},
    {x:2220,y:280,w:185,h:28},
    {x:2650,y:330,w:165,h:28}
  ];

  spikes = [
    {x:690,y:500,w:55,h:30},
    {x:1450,y:500,w:70,h:30},
    {x:1995,y:500,w:70,h:30},
    {x:2890,y:500,w:70,h:30}
  ];

  enemies = [
    makeZetsu(465,418,380,610,"zetsu"),
    makeZetsu(835,373,760,1000,"fast"),
    makeZetsu(1190,320,1130,1400,"zetsu"),
    makeZetsu(1705,418,1550,1950,"spore"),
    makeZetsu(2215,355,2090,2350,"fast"),
    makeZetsu(2550,418,2460,2840,"spore")
  ];

  boss = makeIsshiki(3180,315);
  enemies.push(boss);

  particles = [];
  slashes = [];
  flames = [];
  lightning = [];
  arrows = [];
  cubes = [];
  rods = [];
  orbs = [];
  texts = [];
  leaves = [];

  for(let i=0;i<95;i++){
    leaves.push({
      x:Math.random()*world.width,
      y:Math.random()*H,
      vx:-0.25-Math.random()*0.45,
      vy:0.10+Math.random()*0.3,
      s:2+Math.random()*3,
      c:Math.random()>0.5 ? "#6a8d3d" : "#9a6734"
    });
  }
}

function makeZetsu(x,y,min,max,type){
  const d = difficultyData[difficulty];
  const base = {
    zetsu:{w:42,h:60,hp:4,spd:1.1},
    fast:{w:38,h:56,hp:3,spd:1.85},
    spore:{w:46,h:64,hp:5,spd:1.05}
  }[type];

  return {
    kind:"zetsu", type,
    x,y,w:base.w,h:base.h,
    hp:Math.ceil(base.hp*d.enemyHp),
    maxHp:Math.ceil(base.hp*d.enemyHp),
    vx:base.spd*d.enemySpd,
    min,max,
    alive:true,
    hurt:0,
    burn:0,
    shootCd:100+Math.random()*80
  };
}

function makeIsshiki(x,y){
  const d = difficultyData[difficulty];
  return {
    kind:"isshiki", type:"boss",
    x,y,w:84,h:126,
    hp:Math.ceil(34*d.bossHp),
    maxHp:Math.ceil(34*d.bossHp),
    vx:1.2*d.enemySpd,
    min:2980,max:3500,
    alive:true,
    hurt:0,
    burn:0,
    shootCd:70,
    cubeCd:150,
    teleportCd:190,
    phase:1
  };
}

function hitbox(a,b){
  return a.x < b.x+b.w && a.x+a.w > b.x && a.y < b.y+b.h && a.y+a.h > b.y;
}

function addText(x,y,t,c){
  texts.push({x,y,t,c,life:46,vy:-1.1});
}

function particlesAt(x,y,c,n=12,p=5){
  for(let i=0;i<n;i++){
    particles.push({
      x,y,
      vx:(Math.random()-0.5)*p,
      vy:(Math.random()-0.85)*p,
      life:30+Math.random()*28,
      c,
      s:2+Math.random()*3
    });
  }
}

function gainSusanoo(v){
  player.susanoo = Math.min(player.susanooMax, player.susanoo + v);
}

function hurtEnemy(e,dmg,c,knock=true){
  if(!e.alive) return;

  if(player.susanooActive && dmg < 5) dmg += 1;

  e.hp -= dmg;
  e.hurt = 12;

  if(knock) e.x += player.facing * 14;

  particlesAt(e.x+e.w/2,e.y+e.h/2,c,16,5);
  addText(e.x,e.y-8,"-"+dmg,c);

  player.chakra = Math.min(player.maxChakra, player.chakra + 4);

  if(e.hp <= 0){
    e.alive = false;
    particlesAt(e.x+e.w/2,e.y+e.h/2,e.kind==="isshiki" ? "#f7df9f" : "#eaffdd",48,9);
    orbs.push({x:e.x+e.w/2,y:e.y,vy:-4,col:false});

    if(e.kind !== "isshiki") gainSusanoo(25);

    if(e.kind === "isshiki"){
      state = "victoryCutscene";
      victoryTimer = 0;
      stormFlash = 18;
      beep(600,0.18,"sawtooth",0.045);
      beep(880,0.28,"triangle",0.04);
    }
  }
}

function hurtPlayer(dmg,dir){
  if(player.inv > 0 || state !== "playing") return;

  const dif = difficultyData[difficulty];
  dmg = Math.ceil(dmg * dif.dmg);

  if(player.susanooActive) dmg = Math.max(1, dmg-1);

  player.hp -= dmg;
  player.inv = player.susanooActive ? 45 : 70;
  player.vx = dir*8;
  player.vy = -8;

  particlesAt(player.x+player.w/2,player.y+player.h/2,"#ff4d6d",24,6);
  addText(player.x,player.y-12,"-"+dmg,"#ff4d6d");
  beep(120,0.12,"sawtooth",0.05);

  if(player.hp <= 0){
    player.hp = 0;
    state = "gameover";
  }
}

function sword(){
  if(player.atkCd > 0) return;

  player.atkCd = player.susanooActive ? 16 : 23;
  let reach = player.susanooActive ? 128 : 76;
  let box = {
    x: player.x + (player.facing>0 ? player.w : -reach),
    y: player.y + (player.susanooActive ? -10 : 16),
    w: reach,
    h: player.susanooActive ? 82 : 42,
    life:10,
    dir:player.facing,
    sus:player.susanooActive
  };
  slashes.push(box);
  beep(player.susanooActive ? 360 : 290,0.07,"triangle",0.04);

  for(let e of enemies){
    if(e.alive && hitbox(box,e)){
      hurtEnemy(e, player.susanooActive ? 3 : 1, player.susanooActive ? "#cfa7ff" : "#d6d9ff");
    }
  }
}

function dash(){
  if(player.dashCd > 0 || player.dash > 0) return;
  player.dash = 10;
  player.dashCd = 45;
  player.vx = player.facing*15;
  particlesAt(player.x+player.w/2,player.y+player.h/2,"#9d8cff",22,6);
  beep(240,0.08,"sine",0.04);
}

function amaterasu(){
  if(player.amaCd > 0 || player.chakra < 18) return;
  player.chakra -= 18;
  player.amaCd = 65;
  flames.push({
    x:player.x+player.w/2+player.facing*80,
    y:player.y+38,
    w:62,h:70,
    vx:player.facing*6,
    life:148,
    tick:0,
    enemy:false
  });
  particlesAt(player.x+player.facing*75,player.y+38,"#09090d",30,6);
  beep(90,0.18,"sawtooth",0.04);
}

function kirin(){
  if(player.kirinCd > 0 || player.chakra < 60) return;
  player.chakra -= 60;
  player.kirinCd = 210;
  stormFlash = 18;

  let tx = Math.max(100,Math.min(world.width-100, player.x + player.facing*285));
  lightning.push({x:tx,r:170,life:30,hit:false});
  particlesAt(tx,325,"#bff7ff",65,9);
  beep(720,0.16,"square",0.035);
}

function activateSusanoo(){
  if(player.susanooActive || player.susanoo < player.susanooMax) return;

  player.susanooActive = true;
  player.susanooTimer = player.susanooDuration;
  player.susanoo = 0;
  stormFlash = 10;
  particlesAt(player.x+player.w/2,player.y+player.h/2,"#a970ff",85,10);
  addText(player.x-20, player.y-22, "SUSANOO!", "#d8b4ff");
  beep(180,0.18,"sawtooth",0.05);
  beep(360,0.18,"triangle",0.04);
}

function checkIndraCombo(){
  const seq = ["h","j","k","l"];
  if(combo.length < seq.length) return false;

  let ok = true;
  for(let i=0;i<seq.length;i++){
    if(combo[combo.length-seq.length+i] !== seq[i]) ok = false;
  }
  return ok;
}

function indraArrow(){
  if(!player.susanooActive || player.indraCd > 0 || player.chakra < 35) return;

  player.chakra -= 35;
  player.indraCd = 170;
  stormFlash = 12;

  arrows.push({
    x:player.x+player.w/2,
    y:player.y-34,
    vx:player.facing*18,
    w:150,
    h:38,
    life:90,
    hit:false
  });

  particlesAt(player.x+player.w/2,player.y-20,"#d8b4ff",70,9);
  addText(player.x-25,player.y-50,"ARCO DE INDRA!", "#d8b4ff");
  beep(520,0.12,"sawtooth",0.045);
  beep(1040,0.16,"square",0.035);
}

function updatePlayer(){
  if(state !== "playing") return;

  player.chakra = Math.min(player.maxChakra, player.chakra + 0.035);

  if(player.susanooActive){
    player.susanooTimer--;
    if(player.susanooTimer <= 0){
      player.susanooActive = false;
      particlesAt(player.x+player.w/2,player.y+player.h/2,"#a970ff",35,7);
    }
  }

  if(player.inv>0) player.inv--;
  if(player.atkCd>0) player.atkCd--;
  if(player.dashCd>0) player.dashCd--;
  if(player.dash>0) player.dash--;
  if(player.amaCd>0) player.amaCd--;
  if(player.kirinCd>0) player.kirinCd--;
  if(player.indraCd>0) player.indraCd--;

  let moving = false;

  if(keys["a"]){
    player.vx -= player.speed;
    player.facing = -1;
    moving = true;
  }
  if(keys["d"]){
    player.vx += player.speed;
    player.facing = 1;
    moving = true;
  }

  if(!moving && player.dash <= 0) player.vx *= friction;

  if(player.dash <= 0){
    player.vx = Math.max(-player.maxSpeed, Math.min(player.maxSpeed, player.vx));
  }

  const jumpPressed = pressed["w"] || pressed[" "];

  if(jumpPressed && player.grounded){
    player.vy = player.jump;
    player.grounded = false;
    player.doubleJump = true;
    particlesAt(player.x+player.w/2,player.y+player.h,"#6d6a56",10,4);
    beep(260,0.05,"sine",0.025);
  } else if(jumpPressed && player.doubleJump && !player.grounded){
    player.vy = player.jump*0.86;
    player.doubleJump = false;
    particlesAt(player.x+player.w/2,player.y+player.h/2,"#b6a6ff",16,5);
    beep(390,0.05,"sine",0.025);
  }

  if(pressed["j"]) sword();
  if(pressed["k"]) dash();
  if(pressed["u"]) amaterasu();
  if(pressed["i"]) kirin();
  if(pressed["o"]) activateSusanoo();

  if(checkIndraCombo()){
    indraArrow();
    combo = [];
  }

  if(player.dash <= 0) player.vy += gravity;
  else player.vy *= 0.25;

  player.x += player.vx;
  collideX();

  player.y += player.vy;
  collideY();

  player.x = Math.max(0, Math.min(world.width-player.w, player.x));

  if(player.y > H+250){
    hurtPlayer(2,-player.facing);
    player.x = 80;
    player.y = 260;
    player.vx = 0;
    player.vy = 0;
  }

  for(let s of spikes){
    if(hitbox(player,s)) hurtPlayer(1, player.x < s.x ? -1 : 1);
  }

  if(!bossIntro.done && player.x > 2850){
    state = "bossIntro";
    bossIntro.timer = 0;
    player.vx = 0;
    player.vy = 0;
  }
}

function collideX(){
  for(let p of platforms){
    if(hitbox(player,p)){
      if(player.vx > 0) player.x = p.x-player.w;
      if(player.vx < 0) player.x = p.x+p.w;
      player.vx = 0;
    }
  }
}

function collideY(){
  player.grounded = false;
  for(let p of platforms){
    if(hitbox(player,p)){
      if(player.vy > 0){
        player.y = p.y-player.h;
        player.vy = 0;
        player.grounded = true;
        player.doubleJump = true;
      } else if(player.vy < 0){
        player.y = p.y+p.h;
        player.vy = 0;
      }
    }
  }
}

function updateEnemies(){
  if(state !== "playing") return;

  for(let e of enemies){
    if(!e.alive) continue;

    if(e.hurt>0) e.hurt--;

    if(e.burn>0){
      e.burn--;
      if(e.burn % 24 === 0) hurtEnemy(e,1,"#111",false);
      particlesAt(e.x+e.w/2,e.y+e.h/2,"#050509",2,2.5);
    }

    if(e.kind === "isshiki"){
      updateIsshiki(e);
    } else {
      updateZetsu(e);
    }

    if(hitbox(player,e)){
      hurtPlayer(e.kind==="isshiki" ? 2 : 1, player.x < e.x ? -1 : 1);
    }

    if(player.susanooActive){
      let aura = {x:player.x-55,y:player.y-85,w:player.w+110,h:player.h+120};
      if(hitbox(aura,e) && Math.random()<0.015){
        hurtEnemy(e,1,"#cfa7ff",false);
      }
    }
  }
}

function updateZetsu(e){
  e.x += e.vx;
  if(e.x < e.min || e.x+e.w > e.max) e.vx *= -1;

  let dx = player.x - e.x;
  if((e.type==="fast" || e.type==="spore") && Math.abs(dx)<330){
    e.vx += Math.sign(dx)*0.035;
    e.vx = Math.max(-2.35, Math.min(2.35, e.vx));
  }

  if(e.type==="spore"){
    e.shootCd--;
    if(e.shootCd <= 0 && Math.abs(dx)<380){
      e.shootCd = 120;
      flames.push({
        x:e.x+e.w/2,y:e.y+e.h/2,
        w:38,h:38,
        vx:Math.sign(dx)*3.25,
        life:80,tick:0,enemy:true
      });
      beep(160,0.08,"triangle",0.02);
    }
  }
}

function updateIsshiki(e){
  e.phase = e.hp < e.maxHp*0.5 ? 2 : 1;

  let dx = player.x - e.x;
  e.vx += Math.sign(dx)*0.03*(e.phase===2?1.4:1);
  e.vx = Math.max(-2.4, Math.min(2.4, e.vx));

  e.x += e.vx;

  if(e.x < e.min || e.x+e.w > e.max) e.vx *= -1;

  e.shootCd--;
  e.cubeCd--;
  e.teleportCd--;

  if(e.shootCd <= 0){
    e.shootCd = e.phase===2 ? 38 : 58;
    shootRod(e);
  }

  if(e.cubeCd <= 0){
    e.cubeCd = e.phase===2 ? 115 : 155;
    dropCube(e);
  }

  if(e.teleportCd <= 0){
    e.teleportCd = e.phase===2 ? 120 : 190;
    particlesAt(e.x+e.w/2,e.y+e.h/2,"#f7df9f",28,7);
    e.x = Math.max(e.min, Math.min(e.max, player.x + (Math.random()>0.5 ? 180 : -180)));
    particlesAt(e.x+e.w/2,e.y+e.h/2,"#f7df9f",28,7);
    beep(300,0.1,"square",0.025);
  }
}

function shootRod(e){
  let dir = player.x < e.x ? -1 : 1;
  rods.push({
    x:e.x+e.w/2,
    y:e.y+45,
    w:46,h:8,
    vx:dir*(e.phase===2?7.5:6.2),
    life:120
  });
  beep(210,0.06,"square",0.025);
}

function dropCube(e){
  cubes.push({
    x:player.x-35+Math.random()*70,
    y:-80,
    w:e.phase===2?90:75,
    h:e.phase===2?90:75,
    vy:e.phase===2?6.5:5.5,
    life:200
  });
  stormFlash = 4;
  beep(80,0.16,"sawtooth",0.035);
}

function updateProjectiles(){
  flames = flames.filter(f=>f.life>0);
  for(let f of flames){
    f.x += f.vx;
    f.life--;
    f.tick++;

    let box = {x:f.x-f.w/2,y:f.y-f.h/2,w:f.w,h:f.h};

    if(f.enemy){
      if(hitbox(player,box)){
        hurtPlayer(1, player.x < f.x ? -1 : 1);
        f.life = 0;
      }
    } else {
      for(let e of enemies){
        if(e.alive && hitbox(box,e)){
          e.burn = Math.max(e.burn,100);
          if(f.tick % 18 === 0) hurtEnemy(e,1,"#111",false);
        }
      }
    }
  }

  lightning = lightning.filter(l=>l.life>0);
  for(let l of lightning){
    l.life--;
    if(!l.hit && l.life < 21){
      l.hit = true;
      for(let e of enemies){
        if(!e.alive) continue;
        let cx=e.x+e.w/2, cy=e.y+e.h/2;
        let dist=Math.hypot(cx-l.x, cy-330);
        if(dist<l.r){
          hurtEnemy(e, e.kind==="isshiki"?7:4, "#bff7ff", false);
          e.burn = Math.max(e.burn,35);
        }
      }
    }
  }

  arrows = arrows.filter(a=>a.life>0);
  for(let a of arrows){
    a.x += a.vx;
    a.life--;

    let box = {
      x:a.vx>0 ? a.x : a.x-a.w,
      y:a.y,
      w:a.w,
      h:a.h
    };

    particlesAt(a.x, a.y+a.h/2, "#d8b4ff", 2, 2.5);

    for(let e of enemies){
      if(e.alive && hitbox(box,e)){
        hurtEnemy(e, e.kind==="isshiki" ? 12 : 8, "#d8b4ff", false);
        e.burn = Math.max(e.burn,80);
        a.life = 0;
        stormFlash = 10;
        particlesAt(e.x+e.w/2,e.y+e.h/2,"#d8b4ff",75,10);
        beep(90,0.20,"sawtooth",0.05);
      }
    }
  }

  rods = rods.filter(r=>r.life>0);
  for(let r of rods){
    r.x += r.vx;
    r.life--;
    if(hitbox(player,r)){
      hurtPlayer(1, player.x < r.x ? -1 : 1);
      r.life = 0;
    }
  }

  cubes = cubes.filter(c=>c.life>0);
  for(let c of cubes){
    c.y += c.vy;
    c.life--;

    if(hitbox(player,c)){
      hurtPlayer(2, player.x < c.x ? -1 : 1);
      c.life = 0;
      particlesAt(c.x+c.w/2,c.y+c.h/2,"#d6bc72",35,8);
    }

    if(c.y > 500){
      c.life = 0;
      particlesAt(c.x+c.w/2,500,"#d6bc72",30,7);
    }
  }

  if(stormFlash>0) stormFlash--;
}

function updateOrbs(){
  for(let o of orbs){
    if(o.col) continue;
    o.vy += 0.2;
    o.y += o.vy;
    if(o.y > 460){
      o.y = 460;
      o.vy *= -0.35;
    }
    let box = {x:o.x-16,y:o.y-16,w:46,h:46};
    if(hitbox(player,box)){
      o.col = true;
      player.chakra = Math.min(player.maxChakra, player.chakra+24);
      particlesAt(o.x,o.y,"#b6a6ff",20,5);
      beep(520,0.08,"sine",0.025);
    }
  }
}

function updateEffects(){
  particles = particles.filter(p=>p.life>0);
  for(let p of particles){
    p.x += p.vx;
    p.y += p.vy;
    p.vy += 0.13;
    p.life--;
  }

  slashes = slashes.filter(s=>s.life>0);
  for(let s of slashes) s.life--;

  texts = texts.filter(t=>t.life>0);
  for(let t of texts){
    t.y += t.vy;
    t.life--;
  }

  for(let lf of leaves){
    lf.x += lf.vx;
    lf.y += lf.vy;

    if(lf.x < cam.x-150){
      lf.x = cam.x+W+Math.random()*260;
      lf.y = Math.random()*H;
    }
    if(lf.y > H) lf.y = -20;
  }
}

function updateCamera(){
  if(state === "bossIntro"){
    let tx = boss.x + boss.w/2 - W/2;
    cam.x += (Math.max(0,Math.min(world.width-W,tx))-cam.x)*0.045;
  } else {
    cam.x = player.x + player.w/2 - W/2;
    cam.x = Math.max(0, Math.min(world.width-W, cam.x));
  }
}

function updateCutscenes(){
  if(state === "bossIntro"){
    bossIntro.timer++;
    if(bossIntro.timer % 18 === 0){
      particlesAt(boss.x+boss.w/2,boss.y+boss.h/2,"#f7df9f",10,4);
    }
    if(bossIntro.timer > 260){
      bossIntro.done = true;
      state = "playing";
      stormFlash = 8;
    }
  }

  if(state === "victoryCutscene"){
    victoryTimer++;
    if(victoryTimer % 15 === 0){
      particlesAt(player.x+player.w/2,player.y+player.h/2,"#d8b4ff",14,6);
    }
  }
}

function update(){
  if(state === "menu"){
    pressed = {};
    return;
  }

  updateCutscenes();
  updatePlayer();
  updateEnemies();
  updateProjectiles();
  updateOrbs();
  updateEffects();
  updateCamera();

  pressed = {};
}

function bg(){
  let g = ctx.createLinearGradient(0,0,0,H);
  g.addColorStop(0,"#060716");
  g.addColorStop(0.55,"#121026");
  g.addColorStop(1,"#18171e");
  ctx.fillStyle = g;
  ctx.fillRect(0,0,W,H);

  ctx.fillStyle = "rgba(230,220,255,0.82)";
  ctx.beginPath();
  ctx.arc(835,72,37,0,Math.PI*2);
  ctx.fill();
  ctx.fillStyle = "rgba(75,65,125,0.42)";
  ctx.beginPath();
  ctx.arc(855,62,37,0,Math.PI*2);
  ctx.fill();

  ctx.save();
  ctx.translate(-cam.x*0.14,0);
  for(let i=0;i<11;i++){
    let x=i*430;
    ctx.fillStyle=i%2?"#101224":"#15172a";
    ctx.beginPath();
    ctx.moveTo(x-140,480);
    ctx.lineTo(x+135,150);
    ctx.lineTo(x+410,480);
    ctx.closePath();
    ctx.fill();
  }
  ctx.restore();

  ctx.save();
  ctx.translate(-cam.x*0.32,0);
  for(let i=0;i<25;i++){
    let x=i*175, y=245+(i%3)*16;
    ctx.fillStyle=i%2?"#141426":"#18182b";
    ctx.fillRect(x,y,86,235);

    ctx.fillStyle="#2a2744";
    ctx.beginPath();
    ctx.moveTo(x-8,y);
    ctx.lineTo(x+43,y-40);
    ctx.lineTo(x+94,y);
    ctx.closePath();
    ctx.fill();

    ctx.fillStyle="rgba(230,180,70,0.17)";
    ctx.fillRect(x+20,y+58,12,25);
    ctx.fillRect(x+54,y+118,12,25);
  }
  ctx.restore();

  ctx.save();
  ctx.translate(-cam.x*0.50,0);
  for(let i=0;i<46;i++){
    let x=i*92;
    ctx.fillStyle="#101610";
    ctx.fillRect(x+20,285,17,200);
    ctx.fillStyle="#172514";
    ctx.beginPath();
    ctx.arc(x+28,265,45,0,Math.PI*2);
    ctx.fill();
    ctx.fillStyle="#1f331b";
    ctx.beginPath();
    ctx.arc(x+7,297,35,0,Math.PI*2);
    ctx.arc(x+50,297,37,0,Math.PI*2);
    ctx.fill();
  }
  ctx.restore();

  ctx.save();
  ctx.translate(-cam.x*0.15,0);
  for(let lf of leaves){
    ctx.fillStyle = lf.c;
    ctx.fillRect(lf.x,lf.y,lf.s,lf.s+2);
  }
  ctx.restore();

  if(stormFlash>0){
    ctx.fillStyle = "rgba(190,245,255,"+(stormFlash/34)+")";
    ctx.fillRect(0,0,W,H);
  }
}

function drawPlatforms(){
  ctx.save();
  ctx.translate(-cam.x,0);

  for(let p of platforms){
    ctx.fillStyle="#303035";
    ctx.fillRect(p.x,p.y,p.w,p.h);
    ctx.fillStyle="#686348";
    ctx.fillRect(p.x,p.y,p.w,8);
    ctx.fillStyle="#24231f";
    ctx.fillRect(p.x,p.y+p.h-13,p.w,13);

    ctx.fillStyle="#405c2d";
    for(let i=0;i<p.w;i+=16){
      ctx.fillRect(p.x+i,p.y-5,10,7);
    }
  }

  for(let gx of [1010,1950,2890]){
    ctx.fillStyle="#8c2525";
    ctx.fillRect(gx,278,20,192);
    ctx.fillRect(gx+108,278,20,192);
    ctx.fillRect(gx-25,265,176,20);
    ctx.fillRect(gx-48,238,222,23);
    ctx.fillStyle="#2c1111";
    ctx.fillRect(gx-58,230,242,9);
  }

  for(let sp of spikes){
    ctx.fillStyle="#5a332a";
    let count=Math.floor(sp.w/18);
    for(let i=0;i<count;i++){
      ctx.beginPath();
      ctx.moveTo(sp.x+i*18,sp.y+sp.h);
      ctx.lineTo(sp.x+i*18+9,sp.y);
      ctx.lineTo(sp.x+i*18+18,sp.y+sp.h);
      ctx.closePath();
      ctx.fill();
    }
  }

  ctx.restore();
}

function rounded(x,y,w,h,r,c){
  ctx.fillStyle=c;
  if(ctx.roundRect){
    ctx.beginPath();
    ctx.roundRect(x,y,w,h,r);
    ctx.fill();
  } else ctx.fillRect(x,y,w,h);
}

function drawSusanoo(){
  if(!player.susanooActive) return;

  ctx.save();
  ctx.translate(-cam.x,0);

  let x=player.x+player.w/2;
  let y=player.y+player.h/2;

  ctx.globalAlpha=0.36;
  ctx.fillStyle="#7d3cff";
  ctx.beginPath();
  ctx.ellipse(x,y+6,78,112,0,0,Math.PI*2);
  ctx.fill();

  ctx.globalAlpha=0.58;
  ctx.strokeStyle="#d3a6ff";
  ctx.lineWidth=5;

  ctx.beginPath();
  ctx.arc(x,y-62,40,0,Math.PI*2);
  ctx.stroke();

  ctx.fillStyle="#f1dfff";
  ctx.fillRect(x-20,y-70,12,7);
  ctx.fillRect(x+8,y-70,12,7);

  ctx.strokeStyle="#c084ff";
  ctx.lineWidth=4;
  for(let i=0;i<4;i++){
    ctx.beginPath();
    ctx.arc(x,y-18+i*23,60-i*5,Math.PI*1.08,Math.PI*1.92);
    ctx.stroke();
  }

  ctx.beginPath();
  ctx.moveTo(x-48,y-22);
  ctx.lineTo(x-105,y+28);
  ctx.moveTo(x+48,y-22);
  ctx.lineTo(x+105,y+28);
  ctx.stroke();

  // Arco espiritual quando combo ainda está possível/e no Susanoo
  ctx.strokeStyle="#e9d5ff";
  ctx.lineWidth=6;
  ctx.beginPath();
  ctx.arc(x + player.facing*80, y-20, 60, -1.25, 1.25);
  ctx.stroke();

  ctx.beginPath();
  ctx.moveTo(x + player.facing*40, y-20);
  ctx.lineTo(x + player.facing*135, y-20);
  ctx.stroke();

  ctx.globalAlpha=1;
  ctx.restore();
}

function drawSasuke(){
  ctx.save();
  ctx.translate(-cam.x,0);

  if(player.inv>0 && Math.floor(player.inv/5)%2===0) ctx.globalAlpha=0.45;

  let x=player.x, y=player.y;

  ctx.fillStyle="rgba(0,0,0,0.55)";
  ctx.beginPath();
  ctx.ellipse(x+player.w/2,y+player.h+5,30,8,0,0,Math.PI*2);
  ctx.fill();

  if(player.chakra>30 || player.susanooActive){
    ctx.strokeStyle=player.susanooActive?"rgba(180,110,255,0.75)":"rgba(140,90,255,0.35)";
    ctx.lineWidth=player.susanooActive?3:2;
    ctx.beginPath();
    ctx.arc(x+player.w/2,y+player.h/2,40+Math.sin(Date.now()/120)*4,0,Math.PI*2);
    ctx.stroke();
  }

  ctx.fillStyle="#11111d";
  ctx.beginPath();
  ctx.moveTo(x+5,y+22);
  ctx.lineTo(x-10,y+70);
  ctx.lineTo(x+31,y+68);
  ctx.lineTo(x+36,y+22);
  ctx.closePath();
  ctx.fill();

  rounded(x+8,y+24,24,35,4,"#d7d4e6");
  rounded(x+5,y+38,30,23,4,"#35304f");

  ctx.strokeStyle="#b9adc8";
  ctx.lineWidth=4;
  ctx.beginPath();
  ctx.moveTo(x+4,y+48);
  ctx.bezierCurveTo(x+11,y+55,x+27,y+55,x+36,y+48);
  ctx.stroke();

  ctx.fillStyle="#b9adc8";
  ctx.beginPath();
  ctx.arc(x+20,y+52,5,0,Math.PI*2);
  ctx.fill();

  rounded(x+8,y+60,9,14,2,"#171522");
  rounded(x+24,y+60,9,14,2,"#171522");

  rounded(x+4,y+4,32,28,7,"#ddddea");

  ctx.fillStyle="#05050a";
  ctx.beginPath();
  ctx.moveTo(x+3,y+10);
  ctx.lineTo(x-9,y-9);
  ctx.lineTo(x+6,y+0);
  ctx.lineTo(x+10,y-17);
  ctx.lineTo(x+18,y+0);
  ctx.lineTo(x+27,y-16);
  ctx.lineTo(x+31,y+2);
  ctx.lineTo(x+43,y-7);
  ctx.lineTo(x+37,y+15);
  ctx.lineTo(x+24,y+8);
  ctx.lineTo(x+11,y+13);
  ctx.closePath();
  ctx.fill();

  ctx.fillStyle="#d71732";
  ctx.fillRect(x+11,y+16,6,5);

  ctx.fillStyle="#b6a6ff";
  ctx.beginPath();
  ctx.arc(x+27,y+18,4,0,Math.PI*2);
  ctx.fill();

  ctx.strokeStyle="#5e45b8";
  ctx.lineWidth=1;
  ctx.beginPath();
  ctx.arc(x+27,y+18,7,0,Math.PI*2);
  ctx.stroke();
  ctx.beginPath();
  ctx.arc(x+27,y+18,4,0,Math.PI*2);
  ctx.stroke();

  ctx.strokeStyle="#cfd4ff";
  ctx.lineWidth=3;
  ctx.beginPath();
  ctx.moveTo(x+player.w/2,y+36);
  ctx.lineTo(x+player.w/2+player.facing*35,y+18);
  ctx.stroke();

  if(player.dash>0){
    ctx.strokeStyle="rgba(160,145,255,0.7)";
    ctx.lineWidth=4;
    ctx.beginPath();
    ctx.moveTo(x-player.facing*25,y+30);
    ctx.lineTo(x-player.facing*90,y+42);
    ctx.stroke();
  }

  ctx.globalAlpha=1;
  ctx.restore();
}

function drawEnemies(){
  ctx.save();
  ctx.translate(-cam.x,0);

  for(let e of enemies){
    if(!e.alive) continue;

    if(e.kind==="isshiki") drawIsshiki(e);
    else drawZetsu(e);
  }

  ctx.restore();
}

function drawZetsu(e){
  let x=e.x,y=e.y;
  ctx.globalAlpha=e.hurt>0?0.55:1;

  ctx.fillStyle="rgba(0,0,0,0.5)";
  ctx.beginPath();
  ctx.ellipse(x+e.w/2,y+e.h+6,e.w/1.15,8,0,0,Math.PI*2);
  ctx.fill();

  rounded(x,y,e.w,e.h,16,e.burn>0?"#c9c3d9":"#f0f3e5");

  ctx.fillStyle="#567f3c";
  ctx.beginPath();
  ctx.moveTo(x+e.w/2,y+6);
  ctx.lineTo(x-18,y+e.h*0.26);
  ctx.lineTo(x+6,y+e.h*0.60);
  ctx.lineTo(x+e.w/2,y+e.h*0.45);
  ctx.closePath();
  ctx.fill();

  ctx.beginPath();
  ctx.moveTo(x+e.w/2,y+6);
  ctx.lineTo(x+e.w+18,y+e.h*0.26);
  ctx.lineTo(x+e.w-6,y+e.h*0.60);
  ctx.lineTo(x+e.w/2,y+e.h*0.45);
  ctx.closePath();
  ctx.fill();

  ctx.strokeStyle="#365d2a";
  ctx.lineWidth=2;
  ctx.beginPath();
  ctx.moveTo(x+e.w/2,y+8);
  ctx.lineTo(x+e.w/2,y+e.h-8);
  ctx.moveTo(x+e.w/2,y+25);
  ctx.lineTo(x+8,y+e.h*0.56);
  ctx.moveTo(x+e.w/2,y+25);
  ctx.lineTo(x+e.w-8,y+e.h*0.56);
  ctx.stroke();

  ctx.fillStyle="#101010";
  ctx.fillRect(x+e.w*0.28,y+e.h*0.33,5,7);
  ctx.fillRect(x+e.w*0.62,y+e.h*0.33,5,7);

  if(e.burn>0) drawBurn(x,y,e.w,e.h);

  drawHp(e,"#e9ffd4");
  ctx.globalAlpha=1;
}

function drawIsshiki(e){
  let x=e.x,y=e.y;
  ctx.globalAlpha=e.hurt>0?0.55:1;

  ctx.fillStyle="rgba(0,0,0,0.55)";
  ctx.beginPath();
  ctx.ellipse(x+e.w/2,y+e.h+7,e.w/1.1,9,0,0,Math.PI*2);
  ctx.fill();

  // manto branco
  rounded(x+12,y+28,e.w-24,e.h-28,10,"#e8e1c9");

  // túnica preta
  rounded(x+25,y+42,e.w-50,e.h-48,8,"#15120e");

  // cabeça
  rounded(x+22,y+8,e.w-44,38,10,"#e5dbc3");

  // chifres
  ctx.strokeStyle="#e5dbc3";
  ctx.lineWidth=6;
  ctx.beginPath();
  ctx.moveTo(x+31,y+12);
  ctx.lineTo(x+9,y-18);
  ctx.moveTo(x+e.w-31,y+12);
  ctx.lineTo(x+e.w-9,y-18);
  ctx.stroke();

  // olhos
  ctx.fillStyle="#f4d36b";
  ctx.beginPath();
  ctx.arc(x+34,y+27,4,0,Math.PI*2);
  ctx.fill();

  ctx.fillStyle="#111";
  ctx.beginPath();
  ctx.arc(x+e.w-34,y+27,4,0,Math.PI*2);
  ctx.fill();

  ctx.strokeStyle="#f4d36b";
  ctx.lineWidth=1;
  for(let i=0;i<3;i++){
    ctx.beginPath();
    ctx.arc(x+34,y+27,7+i*4,0,Math.PI*2);
    ctx.stroke();
  }

  // marcas/manto
  ctx.strokeStyle="#d6bc72";
  ctx.lineWidth=3;
  ctx.beginPath();
  ctx.moveTo(x+e.w/2,y+45);
  ctx.lineTo(x+e.w/2,y+e.h-8);
  ctx.moveTo(x+26,y+62);
  ctx.lineTo(x+e.w-26,y+62);
  ctx.stroke();

  // cajado/haste
  ctx.strokeStyle="#1a1a1a";
  ctx.lineWidth=5;
  ctx.beginPath();
  ctx.moveTo(x+e.w-12,y+40);
  ctx.lineTo(x+e.w+8,y+110);
  ctx.stroke();

  // aura
  ctx.globalAlpha=0.32;
  ctx.strokeStyle=e.phase===2?"#ffdf7d":"#d6bc72";
  ctx.lineWidth=4;
  ctx.beginPath();
  ctx.arc(x+e.w/2,y+e.h/2,78+Math.sin(Date.now()/150)*6,0,Math.PI*2);
  ctx.stroke();
  ctx.globalAlpha=e.hurt>0?0.55:1;

  if(e.burn>0) drawBurn(x,y,e.w,e.h);

  drawHp(e,e.phase===2?"#ffdf7d":"#d6bc72");
  ctx.globalAlpha=1;
}

function drawBurn(x,y,w,h){
  ctx.fillStyle="#050509";
  for(let i=0;i<6;i++){
    let fx=x+6+i*(w/6);
    let fy=y+h-8-Math.random()*30;
    ctx.beginPath();
    ctx.moveTo(fx,fy+18);
    ctx.lineTo(fx+6,fy-12);
    ctx.lineTo(fx+12,fy+18);
    ctx.closePath();
    ctx.fill();
  }
}

function drawHp(e,c){
  ctx.fillStyle="#111";
  ctx.fillRect(e.x,e.y-15,e.w,6);
  ctx.fillStyle=c;
  ctx.fillRect(e.x,e.y-15,e.w*(e.hp/e.maxHp),6);
}

function drawProjectiles(){
  ctx.save();
  ctx.translate(-cam.x,0);

  for(let f of flames){
    ctx.globalAlpha=Math.max(0.25,f.life/148);
    for(let i=0;i<9;i++){
      let hh=(f.enemy?24:46)+Math.random()*18;
      let ww=10+Math.random()*7;
      let px=f.x-f.w/2+i*(f.w/8);
      let by=f.y+f.h/2;
      ctx.fillStyle=f.enemy?"#dfffd0":"#050509";
      ctx.beginPath();
      ctx.moveTo(px,by);
      ctx.lineTo(px+ww/2,by-hh);
      ctx.lineTo(px+ww,by);
      ctx.closePath();
      ctx.fill();

      if(!f.enemy){
        ctx.fillStyle="rgba(93,17,168,0.45)";
        ctx.beginPath();
        ctx.moveTo(px+2,by);
        ctx.lineTo(px+ww/2,by-hh*0.65);
        ctx.lineTo(px+ww-2,by);
        ctx.closePath();
        ctx.fill();
      }
    }
    ctx.globalAlpha=1;
  }

  for(let l of lightning){
    let a=l.life/30;
    ctx.globalAlpha=a;
    ctx.strokeStyle="#e6ffff";
    ctx.lineWidth=7;
    ctx.beginPath();
    let x=l.x,y=0;
    ctx.moveTo(x,y);
    for(let i=0;i<11;i++){
      x+=(Math.random()-0.5)*80;
      y+=44;
      ctx.lineTo(x,y);
    }
    ctx.stroke();

    ctx.fillStyle="rgba(190,247,255,0.18)";
    ctx.beginPath();
    ctx.arc(l.x,330,l.r,0,Math.PI*2);
    ctx.fill();
    ctx.globalAlpha=1;
  }

  for(let a of arrows){
    ctx.globalAlpha=Math.max(0.4,a.life/90);
    ctx.strokeStyle="#d8b4ff";
    ctx.lineWidth=8;
    ctx.beginPath();
    ctx.moveTo(a.x,a.y+a.h/2);
    ctx.lineTo(a.x+(a.vx>0?a.w:-a.w),a.y+a.h/2);
    ctx.stroke();

    ctx.fillStyle="#ffffff";
    ctx.beginPath();
    let tipX=a.x+(a.vx>0?a.w:-a.w);
    ctx.moveTo(tipX,a.y+a.h/2);
    ctx.lineTo(tipX-(a.vx>0?28:-28),a.y);
    ctx.lineTo(tipX-(a.vx>0?28:-28),a.y+a.h);
    ctx.closePath();
    ctx.fill();
    ctx.globalAlpha=1;
  }

  for(let r of rods){
    ctx.fillStyle="#111";
    ctx.fillRect(r.x,r.y,r.w,r.h);
    ctx.fillStyle="#d6bc72";
    ctx.fillRect(r.x+r.w-8,r.y-2,8,r.h+4);
  }

  for(let c of cubes){
    ctx.fillStyle="rgba(214,188,114,0.35)";
    ctx.fillRect(c.x,c.y,c.w,c.h);
    ctx.strokeStyle="#d6bc72";
    ctx.lineWidth=3;
    ctx.strokeRect(c.x,c.y,c.w,c.h);
    ctx.beginPath();
    ctx.moveTo(c.x,c.y);
    ctx.lineTo(c.x+c.w,c.y+c.h);
    ctx.moveTo(c.x+c.w,c.y);
    ctx.lineTo(c.x,c.y+c.h);
    ctx.stroke();
  }

  ctx.restore();
}

function drawSlashes(){
  ctx.save();
  ctx.translate(-cam.x,0);

  for(let s of slashes){
    ctx.globalAlpha=s.life/10;
    ctx.strokeStyle=s.sus?"#cfa7ff":"#d6d9ff";
    ctx.lineWidth=s.sus?8:5;
    ctx.beginPath();
    if(s.dir>0){
      ctx.arc(s.x+18,s.y+25,s.sus?68:42,-0.9,0.85);
    } else {
      ctx.arc(s.x+s.w-18,s.y+25,s.sus?68:42,Math.PI-0.85,Math.PI+0.9);
    }
    ctx.stroke();
  }

  ctx.globalAlpha=1;
  ctx.restore();
}

function drawOrbsParticlesTexts(){
  ctx.save();
  ctx.translate(-cam.x,0);

  for(let o of orbs){
    if(o.col) continue;
    ctx.fillStyle="#b6a6ff";
    ctx.beginPath();
    ctx.arc(o.x,o.y,8,0,Math.PI*2);
    ctx.fill();
    ctx.strokeStyle="rgba(185,165,255,0.55)";
    ctx.lineWidth=2;
    ctx.beginPath();
    ctx.arc(o.x,o.y,15,0,Math.PI*2);
    ctx.stroke();
  }

  for(let p of particles){
    ctx.globalAlpha=Math.max(0,p.life/56);
    ctx.fillStyle=p.c;
    ctx.fillRect(p.x,p.y,p.s,p.s);
  }
  ctx.globalAlpha=1;

  ctx.font="bold 16px Arial";
  for(let t of texts){
    ctx.globalAlpha=t.life/46;
    ctx.fillStyle=t.c;
    ctx.fillText(t.t,t.x,t.y);
  }
  ctx.globalAlpha=1;

  ctx.restore();
}

function hud(){
  ctx.fillStyle="rgba(0,0,0,0.56)";
  ctx.fillRect(15,15,395,152);

  for(let i=0;i<player.maxHp;i++){
    ctx.fillStyle=i<player.hp?"#e8e6ff":"#33313f";
    ctx.beginPath();
    ctx.arc(36+i*24,40,9,0,Math.PI*2);
    ctx.fill();
    ctx.strokeStyle="#111";
    ctx.stroke();
  }

  bar(30,68,235,13,player.chakra/player.maxChakra,"#8f7dff","#5b50a8");
  ctx.fillStyle="#ddd";
  ctx.font="12px Arial";
  ctx.fillText("CHAKRA",275,79);

  let sr=player.susanooActive ? player.susanooTimer/player.susanooDuration : player.susanoo/player.susanooMax;
  bar(30,96,235,14,sr,player.susanooActive?"#cfa7ff":"#7d3cff","#9f70ff");
  ctx.fillText(player.susanooActive?"SUSANOO ATIVO":"ENERGIA SUSANOO",275,107);

  ctx.fillStyle=player.amaCd<=0 && player.chakra>=18?"#d7c6ff":"#777";
  ctx.fillText("U Amaterasu",30,132);

  ctx.fillStyle=player.kirinCd<=0 && player.chakra>=60?"#bff7ff":"#777";
  ctx.fillText("I Kirin",132,132);

  ctx.fillStyle=player.susanoo>=player.susanooMax && !player.susanooActive?"#cfa7ff":"#777";
  ctx.fillText("O Susanoo",205,132);

  ctx.fillStyle=player.susanooActive && player.indraCd<=0 && player.chakra>=35?"#fff":"#777";
  ctx.fillText("H J K L: Arco de Indra",30,153);

  ctx.fillStyle="#aaa";
  ctx.font="13px Arial";
  ctx.fillText("Dificuldade: "+difficultyData[difficulty].name, 740, 28);

  minimap();
}

function bar(x,y,w,h,r,c,b){
  ctx.fillStyle="#111";
  ctx.fillRect(x,y,w,h);
  ctx.fillStyle=c;
  ctx.fillRect(x,y,w*Math.max(0,Math.min(1,r)),h);
  ctx.strokeStyle=b;
  ctx.strokeRect(x,y,w,h);
}

function minimap(){
  ctx.fillStyle="rgba(0,0,0,0.38)";
  ctx.fillRect(660,512,285,22);
  ctx.strokeStyle="#555";
  ctx.strokeRect(660,512,285,22);

  let px=660+(player.x/world.width)*285;
  ctx.fillStyle="#b6a6ff";
  ctx.fillRect(px,515,5,16);

  for(let e of enemies){
    if(!e.alive) continue;
    let ex=660+(e.x/world.width)*285;
    ctx.fillStyle=e.kind==="isshiki"?"#d6bc72":"#f4f5e8";
    ctx.fillRect(ex,520,4,8);
  }
}

function menu(){
  bg();

  ctx.fillStyle="rgba(0,0,0,0.58)";
  ctx.fillRect(0,0,W,H);

  ctx.textAlign="center";
  ctx.fillStyle="#d8b4ff";
  ctx.font="bold 44px Arial";
  ctx.fillText("SASUKE: SOMBRAS DO ŌTSUTSUKI", W/2, 130);

  ctx.fillStyle="#ddd";
  ctx.font="18px Arial";
  ctx.fillText("Escolha a dificuldade para iniciar", W/2, 180);

  menuButton(340,245,300,50,"1  FÁCIL");
  menuButton(340,310,300,50,"2  NORMAL");
  menuButton(340,375,300,50,"3  DIFÍCIL");

  ctx.fillStyle="#aaa";
  ctx.font="14px Arial";
  ctx.fillText("Clique em uma opção ou pressione 1, 2 ou 3", W/2, 455);
  ctx.fillText("Áudio será ativado após o início do jogo", W/2, 480);

  ctx.textAlign="left";
}

function menuButton(x,y,w,h,text){
  ctx.fillStyle="rgba(35,25,60,0.92)";
  ctx.fillRect(x,y,w,h);
  ctx.strokeStyle="#9f70ff";
  ctx.lineWidth=2;
  ctx.strokeRect(x,y,w,h);
  ctx.fillStyle="#fff";
  ctx.font="bold 20px Arial";
  ctx.fillText(text,x+w/2,y+32);
}

function cutsceneBoss(){
  ctx.fillStyle="rgba(0,0,0,0.45)";
  ctx.fillRect(0,0,W,H);

  ctx.fillStyle="rgba(0,0,0,0.84)";
  ctx.fillRect(0,H-145,W,145);

  ctx.strokeStyle="#d6bc72";
  ctx.lineWidth=2;
  ctx.strokeRect(26,H-128,W-52,105);

  ctx.fillStyle="#f7df9f";
  ctx.font="bold 22px Arial";

  let t = bossIntro.timer;
  let msg = "";

  if(t < 80) msg = "O espaço parece se dobrar à frente...";
  else if(t < 165) msg = "Ishiki: Um humano usando poder ocular contra um Ōtsutsuki?";
  else msg = "Sasuke: Eu não preciso vencer fácil. Só preciso acertar uma vez.";

  ctx.fillText(msg,54,H-82);

  ctx.fillStyle="#aaa";
  ctx.font="14px Arial";
  ctx.fillText("Cutscene de chefe",54,H-50);
}

function victoryCutscene(){
  ctx.fillStyle="rgba(0,0,0,0.72)";
  ctx.fillRect(0,0,W,H);

  ctx.textAlign="center";
  ctx.fillStyle="#d8b4ff";
  ctx.font="bold 46px Arial";

  if(victoryTimer < 100){
    ctx.fillText("A FLECHA DE INDRA ECOA...",W/2,H/2-35);
  } else if(victoryTimer < 210){
    ctx.fillStyle="#f7df9f";
    ctx.fillText("ISHIKI FOI DERROTADO",W/2,H/2-35);
  } else {
    ctx.fillStyle="#bff7ff";
    ctx.fillText("VITÓRIA!",W/2,H/2-35);
    ctx.fillStyle="#ddd";
    ctx.font="22px Arial";
    ctx.fillText("Sasuke atravessou a dimensão inimiga.",W/2,H/2+8);
    ctx.fillText("Pressione R para voltar ao menu.",W/2,H/2+45);
  }

  ctx.textAlign="left";
}

function gameOverScreen(){
  ctx.fillStyle="rgba(0,0,0,0.74)";
  ctx.fillRect(0,0,W,H);

  ctx.textAlign="center";
  ctx.fillStyle="#d7c6ff";
  ctx.font="bold 46px Arial";
  ctx.fillText("VOCÊ FOI DERROTADO", W/2, H/2-25);

  ctx.fillStyle="#ddd";
  ctx.font="22px Arial";
  ctx.fillText("Pressione R para voltar ao menu", W/2, H/2+25);
  ctx.textAlign="left";
}

function draw(){
  if(state === "menu"){
    menu();
    return;
  }

  ctx.clearRect(0,0,W,H);
  bg();
  drawPlatforms();
  drawProjectiles();
  drawSlashes();
  drawSusanoo();
  drawEnemies();
  drawSasuke();
  drawOrbsParticlesTexts();
  hud();

  if(state === "bossIntro") cutsceneBoss();
  if(state === "victoryCutscene") victoryCutscene();
  if(state === "gameover") gameOverScreen();
}

function loop(){
  update();
  draw();
  requestAnimationFrame(loop);
}

resetGame();
state = "menu";
loop();
</script>
""")